## Airflow mini-demo (in one notebook)

This notebook shows a simple end-to-end Airflow demo:

- **Installation** (in a local venv using Python 3.12)
- **DAG definition**
- **PythonOperator** + **BashOperator**
- **Task dependencies**
- **Cron schedule**
- **Airflow UI** startup

Use case: a tiny “ETL” pipeline that generates a CSV, transforms it, then prints a completion message.


In [2]:
# Check Python version(s)
!python3 --version
!python3.12 --version


Python 3.13.7
Python 3.12.11


In [3]:
# Create a dedicated Airflow venv on Python 3.12 (Airflow does not support Python 3.13)
!python3.12 -m venv ./airflow_venv

# Upgrade pip tooling inside the venv
!./airflow_venv/bin/python -m pip install -U pip setuptools wheel

# Install Airflow (pinned + constraints)
!./airflow_venv/bin/python -m pip install "apache-airflow==2.10.5" \
  --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-2.10.5/constraints-3.12.txt"

# Sanity check
!./airflow_venv/bin/airflow version


2.10.5


In [4]:
import os
from pathlib import Path

AIRFLOW_HOME = Path("./airflow_home").resolve()
DAGS_DIR = AIRFLOW_HOME / "dags"

os.environ["AIRFLOW_HOME"] = str(AIRFLOW_HOME)
DAGS_DIR.mkdir(parents=True, exist_ok=True)

str(AIRFLOW_HOME)


'/Users/navalyemul/Documents/uptime/day3_rag/airflow_home'

In [5]:
# Initialize the Airflow metadata DB (SQLite) + create an admin user
!./airflow_venv/bin/airflow db init
!./airflow_venv/bin/airflow users create \
  --username admin \
  --firstname Admin \
  --lastname User \
  --role Admin \
  --email admin@example.com \
  --password admin


/Users/navalyemul/Documents/uptime/day3_rag/airflow_venv/lib/python3.12/site-packages/airflow/utils/providers_configuration_loader.py:55 DeprecationWarning: `db init` is deprecated.  Use `db migrate` instead to migrate the db and/or airflow connections create-default-connections to create the default connections
DB: sqlite:////Users/navalyemul/Documents/uptime/day3_rag/airflow_home/airflow.db
[2026-04-08T13:55:57.317+0530] {migration.py:207} INFO - Context impl SQLiteImpl.
[2026-04-08T13:55:57.319+0530] {migration.py:210} INFO - Will assume non-transactional DDL.
[2026-04-08T13:55:57.436+0530] {migration.py:207} INFO - Context impl SQLiteImpl.
[2026-04-08T13:55:57.436+0530] {migration.py:210} INFO - Will assume non-transactional DDL.
[2026-04-08T13:55:57.437+0530] {migration.py:207} INFO - Context impl SQLiteImpl.
[2026-04-08T13:55:57.437+0530] {migration.py:210} INFO - Will assume non-transactional DDL.
[2026-04-08T13:55:57.438+0530] {db.py:1675} INFO - Creating tables
INFO  [alembic.

In [6]:
dag_path = DAGS_DIR / "sales_pipeline_demo.py"

dag_code = '''
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import csv

from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.operators.python import PythonOperator


def generate_sales(**context):
    out = Path("/tmp/raw_sales.csv")
    rows = [
        {"order_id": 101, "amount": 500},
        {"order_id": 102, "amount": 700},
        {"order_id": 103, "amount": 900},
    ]
    with out.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["order_id", "amount"])
        w.writeheader()
        w.writerows(rows)


def add_tax(**context):
    inp = Path("/tmp/raw_sales.csv")
    out = Path("/tmp/final_sales.csv")

    with inp.open() as f_in, out.open("w", newline="") as f_out:
        r = csv.DictReader(f_in)
        w = csv.DictWriter(f_out, fieldnames=["order_id", "amount", "tax", "total"])
        w.writeheader()

        for row in r:
            amount = float(row["amount"])
            tax = amount * 0.18
            total = amount + tax
            w.writerow(
                {
                    "order_id": int(row["order_id"]),
                    "amount": amount,
                    "tax": tax,
                    "total": total,
                }
            )


with DAG(
    dag_id="sales_pipeline_demo",
    start_date=datetime(2024, 1, 1),
    schedule="*/2 * * * *",  # cron: every 2 minutes
    catchup=False,
    tags=["demo"],
) as dag:
    t1 = PythonOperator(task_id="generate_sales", python_callable=generate_sales)

    t2 = PythonOperator(task_id="add_tax", python_callable=add_tax)

    t3 = BashOperator(task_id="done", bash_command="echo Pipeline finished")

    # Task dependencies
    t1 >> t2 >> t3
'''

dag_path.write_text(dag_code)
str(dag_path)


'/Users/navalyemul/Documents/uptime/day3_rag/airflow_home/dags/sales_pipeline_demo.py'

In [7]:
# Verify Airflow can see the DAG
!./airflow_venv/bin/airflow dags list | head -n 30

# Quick local test run (doesn't require the UI)
!./airflow_venv/bin/airflow dags test sales_pipeline_demo 2026-01-01


/Users/navalyemul/Documents/uptime/day3_rag/airflow_venv/lib/python3.12/site-packages/airflow/cli/commands/dag_command.py:48 UserWarning: Could not import graphviz. Rendering graph to the graphical format will not be possible.
dag_id                                                  | fileloc                                                                                                                  | owners  | is_paused
========================================================+==========================================================================================================================+=========+==========
conditional_dataset_and_time_based_timetable            | /Users/navalyemul/Documents/uptime/day3_rag/airflow_venv/lib/python3.12/site-packages/airflow/example_dags/example_datas | airflow | None     
                                                        | ets.py                                                                                                          

## Start the Airflow UI

Run this in a terminal (recommended) or in a notebook cell (it will keep running):

- **Option A (simplest)**: start scheduler + webserver together

```bash
./airflow_venv/bin/airflow standalone
```

Then open `http://localhost:8080` and login with:

- username: `admin`
- password: `admin`

## What to point out in the demo

- **DAG file**: created at `./airflow_home/dags/sales_pipeline_demo.py`
- **Cron**: `*/2 * * * *` means “every 2 minutes”
- **Operators**:
  - `PythonOperator`: generates/transforms CSVs
  - `BashOperator`: prints completion message
- **Dependencies**: `t1 >> t2 >> t3`
